In [14]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json

from pathlib import Path


In [15]:
# ============================================================
# 1. LOAD BOTH RESULT FILES
# ============================================================

csv_a = Path("/Users/larryhh/Documents/PhD/Projects/weight_matrix_informed_circuit_design/results/larryhh-arm64/wi_experiment.csv")
csv_b = Path("/Users/larryhh/Documents/PhD/Projects/weight_matrix_informed_circuit_design/results/root-x86_64/wi_experiment.csv")

df_a = pd.read_csv(csv_a)
df_b = pd.read_csv(csv_b)

print("Loaded files:")
print(f"  A: {csv_a} -> {df_a.shape}")
print(f"  B: {csv_b} -> {df_b.shape}")
print()

print("Column equality check:")
print(df_a.columns.tolist() == df_b.columns.tolist())
print()

if df_a.columns.tolist() != df_b.columns.tolist():
    print("Columns in A not in B:", sorted(set(df_a.columns) - set(df_b.columns)))
    print("Columns in B not in A:", sorted(set(df_b.columns) - set(df_a.columns)))
    raise ValueError("CSV schemas do not match. Fix this before merging.")

# ============================================================
# 2. MERGE + DEDUPLICATE
# ============================================================

df = pd.concat([df_a, df_b], ignore_index=True)
print(f"Concatenated shape: {df.shape}")

# Drop exact duplicate rows first
df = df.drop_duplicates()
print(f"After exact row deduplication: {df.shape}")

# Optional stronger deduplication:
# keep last row for same experimental identity
dedup_cols = [
    "seed",
    "model_type",
    "dataset_name",
    "ansatz",
    "batch_size",
    "epochs",
    "input_dim",
    "depth",
    "num_classes",
]

# Convert datetime so that "keep=last" is meaningful
df["datetime"] = pd.to_datetime(df["datetime"], errors="coerce")
df = df.sort_values("datetime").drop_duplicates(subset=dedup_cols, keep="last")
print(f"After experiment-identity deduplication: {df.shape}")
print()

# ============================================================
# 3. BASIC SANITY CHECKS
# ============================================================

print("Unique datasets:")
print(sorted(df["dataset_name"].dropna().unique()))
print()

print("Unique model types:")
print(sorted(df["model_type"].dropna().unique()))
print()

print("Unique input_dim values:")
print(sorted(df["input_dim"].dropna().unique()))
print()

print("Seeds present:")
print(sorted(df["seed"].dropna().astype(int).unique()))
print()

print("Unique ansatz values:")
for a in sorted(df["ansatz"].dropna().unique()):
    print(" ", a)
print()

# ============================================================
# 4. KEEP QUANTUM RUNS ONLY
# ============================================================

qdf = df[df["model_type"] == "quantum"].copy()
print(f"Quantum-only shape: {qdf.shape}")
print()

# ============================================================
# 5. PARSE LOSS CURVES
# ============================================================

def parse_loss_list(x):
    if pd.isna(x):
        return None
    if isinstance(x, list):
        return x
    if isinstance(x, str):
        try:
            return json.loads(x)
        except Exception:
            return None
    return None

qdf["train_losses_parsed"] = qdf["train_losses"].apply(parse_loss_list)
qdf["val_losses_parsed"] = qdf["val_losses"].apply(parse_loss_list)

# ============================================================
# 6. COVERAGE TABLE: WHAT SETTINGS HAVE HOW MANY RUNS?
# ============================================================

coverage = (
    qdf.groupby(["dataset_name", "input_dim", "ansatz"])
       .agg(
           n_runs=("seed", "count"),
           seeds=("seed", lambda x: sorted(set(int(v) for v in x if pd.notna(v))))
       )
       .reset_index()
       .sort_values(["dataset_name", "input_dim", "ansatz"])
)

print("===== COVERAGE =====")
print(coverage.to_string(index=False))
print()

print("===== COVERAGE COUNTS BY DATASET / INPUT_DIM =====")
print(
    coverage.groupby(["dataset_name", "input_dim"])["n_runs"]
            .agg(["count", "min", "max"])
            .reset_index()
            .to_string(index=False)
)
print()

# ============================================================
# 7. MAIN SUMMARY TABLE
# ============================================================

summary = (
    qdf.groupby(["dataset_name", "input_dim", "ansatz"])
       .agg(
           n_runs=("test_f1", "count"),
           acc_mean=("test_acc", "mean"),
           acc_std=("test_acc", "std"),
           prec_mean=("test_prec", "mean"),
           prec_std=("test_prec", "std"),
           rec_mean=("test_rec", "mean"),
           rec_std=("test_rec", "std"),
           f1_mean=("test_f1", "mean"),
           f1_std=("test_f1", "std"),
           loss_mean=("test_loss", "mean"),
           loss_std=("test_loss", "std"),
           time_mean=("average_time_per_epoch", "mean"),
           time_std=("average_time_per_epoch", "std"),
       )
       .reset_index()
       .sort_values(["dataset_name", "input_dim", "f1_mean"], ascending=[True, True, False])
)

print("===== FULL SUMMARY =====")
print(summary.to_string(index=False))
print()

# ============================================================
# 8. ONLY FULL 5-SEED SETTINGS
# ============================================================

summary5 = summary[summary["n_runs"] == 5].copy()

print("===== 5-SEED SUMMARY ONLY =====")
print(summary5.to_string(index=False))
print()

print("===== 5-SEED COUNTS BY DATASET / INPUT_DIM =====")
print(
    summary5.groupby(["dataset_name", "input_dim"]).size()
            .reset_index(name="num_methods")
            .to_string(index=False)
)
print()

# ============================================================
# 9. PRINT PER-DATASET TABLES
# ============================================================

for ds in sorted(summary5["dataset_name"].unique()):
    print(f"\n===== {ds.upper()} : 5-SEED SETTINGS =====")
    sub = summary5[summary5["dataset_name"] == ds].sort_values(
        ["input_dim", "f1_mean"], ascending=[True, False]
    )
    print(sub.to_string(index=False))

# ============================================================
# 10. TOP METHODS PER DATASET / INPUT_DIM
# ============================================================

topk = (
    summary5.sort_values(["dataset_name", "input_dim", "f1_mean"], ascending=[True, True, False])
            .groupby(["dataset_name", "input_dim"])
            .head(5)
            .copy()
)

print("\n===== TOP 5 METHODS PER DATASET / INPUT_DIM (BY F1) =====")
print(topk.to_string(index=False))
print()

# ============================================================
# 11. SAVE MERGED + SUMMARIZED OUTPUTS
# ============================================================

out_dir = Path("/Users/larryhh/Documents/PhD/Projects/weight_matrix_informed_circuit_design/results/merged_analysis")
out_dir.mkdir(parents=True, exist_ok=True)

merged_csv = out_dir / "wi_experiment_merged.csv"
coverage_csv = out_dir / "coverage.csv"
summary_csv = out_dir / "summary_all.csv"
summary5_csv = out_dir / "summary_5seed.csv"
topk_csv = out_dir / "top5_by_dataset_inputdim.csv"

df.to_csv(merged_csv, index=False)
coverage.to_csv(coverage_csv, index=False)
summary.to_csv(summary_csv, index=False)
summary5.to_csv(summary5_csv, index=False)
topk.to_csv(topk_csv, index=False)

print("Saved:")
print(" ", merged_csv)
print(" ", coverage_csv)
print(" ", summary_csv)
print(" ", summary5_csv)
print(" ", topk_csv)
print()

# ============================================================
# 12. OPTIONAL: BUILD A SMALL PAPER-FACING TABLE CANDIDATE
#     Edit keep_methods / keep_dims after inspecting summary5
# ============================================================

keep_methods = [
    "HEA (y, z)",
    "Random PQC",
    "MPS (d=1)",
    "MPS (d=2)",
    "MPS (d=3)",
    "TTN (d=1)",
    "TTN (d=2)",
    "TTN (d=3)",
    "WI-PQC (E) (greedy)",
    "WI-PQC (E) (greedy, top_half)",
    "WI-PQC (E) (greedy, cosine_norm)",
    "WI-PQC (E) (mwst, cosine_norm)",
    "WI-PQC (W+E, chunking)",
    "WI-PQC (W+E, svd)",
    "WI-PQC (W1+W2+E, chunking)",
    "WI-PQC (W1+W2+E, svd)",
]

paper_candidates = summary5[summary5["ansatz"].isin(keep_methods)].copy()

print("===== PAPER CANDIDATES (5-SEED ONLY) =====")
print(
    paper_candidates.sort_values(["dataset_name", "input_dim", "f1_mean"], ascending=[True, True, False])
                    .to_string(index=False)
)
print()

paper_candidates_csv = out_dir / "paper_candidates_5seed.csv"
paper_candidates.to_csv(paper_candidates_csv, index=False)
print("Saved:", paper_candidates_csv)
print()

# ============================================================
# 13. OPTIONAL: MEAN LOSS CURVE EXTRACTION
#     Change dataset_name / input_dim / ansatz_name as needed
# ============================================================

def mean_curve(curves):
    curves = [c for c in curves if isinstance(c, list) and len(c) > 0]
    if len(curves) == 0:
        return None, None
    min_len = min(len(c) for c in curves)
    arr = np.array([c[:min_len] for c in curves], dtype=float)
    return arr.mean(axis=0), arr.std(axis=0)

dataset_name = "iris"
input_dim = 4
ansatz_name = "WI-PQC (W1+W2+E, chunking)"

subset = qdf[
    (qdf["dataset_name"] == dataset_name) &
    (qdf["input_dim"] == input_dim) &
    (qdf["ansatz"] == ansatz_name)
].copy()

train_mean, train_std = mean_curve(subset["train_losses_parsed"].tolist())
val_mean, val_std = mean_curve(subset["val_losses_parsed"].tolist())

print(f"===== LOSS CURVE CHECK: {dataset_name}, {input_dim}q, {ansatz_name} =====")
print("n_rows:", len(subset))
print("train_mean:", train_mean)
print("val_mean:", val_mean)

# Save one example curve if available
if train_mean is not None and val_mean is not None:
    curve_df = pd.DataFrame({
        "epoch": np.arange(1, len(train_mean) + 1),
        "train_mean": train_mean,
        "train_std": train_std,
        "val_mean": val_mean,
        "val_std": val_std,
    })
    curve_csv = out_dir / f"curve_{dataset_name}_{input_dim}q_{ansatz_name.replace('/', '-').replace(' ', '_')}.csv"
    curve_df.to_csv(curve_csv, index=False)
    print("Saved example curve:", curve_csv)
else:
    print("No valid loss curves found for that selection.")

Loaded files:
  A: /Users/larryhh/Documents/PhD/Projects/weight_matrix_informed_circuit_design/results/larryhh-arm64/wi_experiment.csv -> (416, 28)
  B: /Users/larryhh/Documents/PhD/Projects/weight_matrix_informed_circuit_design/results/root-x86_64/wi_experiment.csv -> (170, 28)

Column equality check:
True

Concatenated shape: (586, 28)
After exact row deduplication: (586, 28)
After experiment-identity deduplication: (564, 28)

Unique datasets:
['diabetes', 'iris', 'wine']

Unique model types:
['classical', 'quantum']

Unique input_dim values:
[4, 6, 8]

Seeds present:
[0, 1, 2, 3, 4]

Unique ansatz values:
  HEA (y, z)
  MPS (d=1)
  MPS (d=2)
  MPS (d=3)
  Random PQC
  TTN (d=1)
  TTN (d=2)
  TTN (d=3)
  WI-PQC (E) (greedy)
  WI-PQC (E) (greedy, cosine_norm)
  WI-PQC (E) (greedy, top_half)
  WI-PQC (E) (mwst, cosine_norm)
  WI-PQC (W+E, chunking)
  WI-PQC (W+E, svd)
  WI-PQC (W1+W2+E, chunking)
  WI-PQC (W1+W2+E, svd)

Quantum-only shape: (530, 28)

===== COVERAGE =====
dataset_name 

In [16]:
print(summary5.to_string(index=False))
print(topk.to_string(index=False))
print(coverage.groupby(["dataset_name", "input_dim"])["n_runs"].agg(["count", "min", "max"]).reset_index().to_string(index=False))

dataset_name  input_dim                           ansatz  n_runs  acc_mean  acc_std  prec_mean  prec_std  rec_mean  rec_std  f1_mean   f1_std  loss_mean  loss_std  time_mean   time_std
    diabetes          4                       Random PQC       5   0.47792 0.050444    0.35886  0.037375   0.65026 0.110947  0.46090 0.053070    0.80244  0.077179    4.01078   2.798080
    diabetes          4                WI-PQC (W+E, svd)       5   0.65196 0.046050    0.50174  0.084353   0.37598 0.040132  0.42858 0.053398    0.60182  0.032431   67.29220  71.999147
    diabetes          4   WI-PQC (E) (mwst, cosine_norm)       5   0.67404 0.042177    0.54428  0.085979   0.34990 0.082932  0.42378 0.080268    0.61738  0.018010   84.13322 110.679835
    diabetes          4           WI-PQC (W+E, chunking)       5   0.66232 0.049888    0.52116  0.086243   0.35758 0.092902  0.41960 0.086912    0.60522  0.031008   69.67990  77.193956
    diabetes          4              WI-PQC (E) (greedy)       5   0.66752 

In [17]:
import pandas as pd
from pathlib import Path

# ============================================================
# LOAD THE MERGED 5-SEED SUMMARY
# ============================================================

summary5_fp = Path("/Users/larryhh/Documents/PhD/Projects/weight_matrix_informed_circuit_design/results/merged_analysis/summary_5seed.csv")
summary5 = pd.read_csv(summary5_fp)

# ------------------------------------------------------------
# Helper predicates
# ------------------------------------------------------------

def is_wi(ansatz: str) -> bool:
    return isinstance(ansatz, str) and ansatz.startswith("WI-PQC")

def is_tn(ansatz: str) -> bool:
    return isinstance(ansatz, str) and (ansatz.startswith("MPS") or ansatz.startswith("TTN"))

def is_hea(ansatz: str) -> bool:
    return ansatz == "HEA (y, z)"

def is_random(ansatz: str) -> bool:
    return ansatz == "Random PQC"

# ------------------------------------------------------------
# Selectors
# ------------------------------------------------------------

def best_row(df: pd.DataFrame, mask_fn):
    sub = df[df["ansatz"].apply(mask_fn)].copy()
    if sub.empty:
        return None
    sub = sub.sort_values(
        ["f1_mean", "loss_mean", "acc_mean"],
        ascending=[False, True, False]
    )
    return sub.iloc[0]

def top_two_wi(df: pd.DataFrame):
    sub = df[df["ansatz"].apply(is_wi)].copy()
    if sub.empty:
        return []
    sub = sub.sort_values(
        ["f1_mean", "loss_mean", "acc_mean"],
        ascending=[False, True, False]
    )
    return [row for _, row in sub.head(2).iterrows()]

# ============================================================
# BUILD CROSS-QUBIT SUMMARY TABLE CONTENT
# ============================================================

rows_cross = []

for (dataset, input_dim), g in summary5.groupby(["dataset_name", "input_dim"], sort=True):
    hea = best_row(g, is_hea)
    wi = best_row(g, is_wi)
    tn = best_row(g, is_tn)
    rnd = best_row(g, is_random)

    rows_cross.append({
        "dataset": dataset,
        "q": int(input_dim),
        "HEA_f1": hea["f1_mean"] if hea is not None else None,
        "HEA_loss": hea["loss_mean"] if hea is not None else None,
        "best_wi_name": wi["ansatz"] if wi is not None else None,
        "best_wi_f1": wi["f1_mean"] if wi is not None else None,
        "best_wi_loss": wi["loss_mean"] if wi is not None else None,
        "best_tn_name": tn["ansatz"] if tn is not None else None,
        "best_tn_f1": tn["f1_mean"] if tn is not None else None,
        "best_tn_loss": tn["loss_mean"] if tn is not None else None,
        "random_f1": rnd["f1_mean"] if rnd is not None else None,
        "random_loss": rnd["loss_mean"] if rnd is not None else None,
    })

cross_df = pd.DataFrame(rows_cross).sort_values(["dataset", "q"])

print("===== CROSS-QUBIT SUMMARY SELECTIONS =====")
print(cross_df.to_string(index=False))
print()

# ============================================================
# BUILD WI-PQC ABLATION TABLE CONTENT
# ============================================================

rows_ablation = []

for (dataset, input_dim), g in summary5.groupby(["dataset_name", "input_dim"], sort=True):
    best_two = top_two_wi(g)
    if len(best_two) == 0:
        continue

    best1 = best_two[0]
    best2 = best_two[1] if len(best_two) > 1 else None

    rows_ablation.append({
        "dataset": dataset,
        "q": int(input_dim),
        "best_wi": best1["ansatz"],
        "best_wi_f1": best1["f1_mean"],
        "best_wi_loss": best1["loss_mean"],
        "runner_up_wi": best2["ansatz"] if best2 is not None else None,
        "runner_up_wi_f1": best2["f1_mean"] if best2 is not None else None,
    })

ablation_df = pd.DataFrame(rows_ablation).sort_values(["dataset", "q"])

print("===== WI-PQC ABLATION SELECTIONS =====")
print(ablation_df.to_string(index=False))
print()

# ============================================================
# LATEX HELPERS
# ============================================================

def fmt_pair(f1, loss):
    if pd.isna(f1) or pd.isna(loss):
        return "---"
    return f"{f1:.3f} ({loss:.3f})"

def latex_escape(s: str) -> str:
    if s is None or pd.isna(s):
        return "---"
    s = s.replace("W1+W2", "W$_1$W$_2$")
    return s

# ------------------------------------------------------------
# Generate Table 1 rows
# ------------------------------------------------------------

print("===== LATEX ROWS: TABLE 1 (CROSS-QUBIT SUMMARY) =====")
for _, r in cross_df.iterrows():
    line = (
        f'{r["dataset"].capitalize()} & {int(r["q"])} '
        f'& {fmt_pair(r["HEA_f1"], r["HEA_loss"])} '
        f'& {latex_escape(r["best_wi_name"])}: {fmt_pair(r["best_wi_f1"], r["best_wi_loss"])} '
        f'& {latex_escape(r["best_tn_name"])}: {fmt_pair(r["best_tn_f1"], r["best_tn_loss"])} '
        f'& {fmt_pair(r["random_f1"], r["random_loss"])} \\\\'
    )
    print(line)
print()

# ------------------------------------------------------------
# Generate Table 2 rows
# ------------------------------------------------------------

print("===== LATEX ROWS: TABLE 2 (WI-PQC ABLATION) =====")
for _, r in ablation_df.iterrows():
    best = f'{latex_escape(r["best_wi"])} & {r["best_wi_f1"]:.3f} & {r["best_wi_loss"]:.3f}'
    if pd.isna(r["runner_up_wi_f1"]):
        runner = "--- & ---"
    else:
        runner = f'{latex_escape(r["runner_up_wi"])} & {r["runner_up_wi_f1"]:.3f}'
    line = f'{r["dataset"].capitalize()} & {int(r["q"])} & {best} & {runner} \\\\'
    print(line)
print()

# ============================================================
# OPTIONAL: SAVE THE SELECTION TABLES
# ============================================================

out_dir = summary5_fp.parent
cross_df.to_csv(out_dir / "table1_cross_qubit_summary_content.csv", index=False)
ablation_df.to_csv(out_dir / "table2_wipqc_ablation_content.csv", index=False)

print("Saved:")
print(" ", out_dir / "table1_cross_qubit_summary_content.csv")
print(" ", out_dir / "table2_wipqc_ablation_content.csv")

===== CROSS-QUBIT SUMMARY SELECTIONS =====
 dataset  q  HEA_f1  HEA_loss                     best_wi_name  best_wi_f1  best_wi_loss best_tn_name  best_tn_f1  best_tn_loss  random_f1  random_loss
diabetes  4 0.40696   0.63082                WI-PQC (W+E, svd)     0.42858       0.60182    MPS (d=2)     0.37140       0.65382    0.46090      0.80244
diabetes  6 0.38894   0.65814 WI-PQC (E) (greedy, cosine_norm)     0.40830       0.65130    MPS (d=3)     0.38632       0.67640    0.44684      0.70792
diabetes  8 0.34908   0.65702    WI-PQC (E) (greedy, top_half)     0.36464       0.67948    MPS (d=2)     0.48270       0.67916    0.39996      0.69456
    iris  4 0.59914   0.91880            WI-PQC (W1+W2+E, svd)     0.62900       0.93464    MPS (d=3)     0.54958       0.98370    0.32286      1.11392
    wine  4 0.57754   0.92942              WI-PQC (E) (greedy)     0.64864       0.89260    TTN (d=2)     0.55888       0.97184    0.39602      1.10012
    wine  6 0.39600   1.06854       WI-PQC (W

In [ ]:
import pandas as pd
from pathlib import Path

# ============================================================
# LOAD SUMMARY
# ============================================================

summary5_fp = Path("/Users/larryhh/Documents/PhD/Projects/weight_matrix_informed_circuit_design/results/merged_analysis/summary_5seed.csv")
summary5 = pd.read_csv(summary5_fp)

# ============================================================
# HELPERS
# ============================================================

def is_wi(ansatz: str) -> bool:
    return isinstance(ansatz, str) and ansatz.startswith("WI-PQC")

def is_mps(ansatz: str) -> bool:
    return isinstance(ansatz, str) and ansatz.startswith("MPS")

def is_ttn(ansatz: str) -> bool:
    return isinstance(ansatz, str) and ansatz.startswith("TTN")

def is_hea(ansatz: str) -> bool:
    return ansatz == "HEA (y, z)"

def is_random(ansatz: str) -> bool:
    return ansatz == "Random PQC"

def best_row(df: pd.DataFrame, mask_fn):
    sub = df[df["ansatz"].apply(mask_fn)].copy()
    if sub.empty:
        return None
    sub = sub.sort_values(
        ["f1_mean", "acc_mean", "loss_mean"],
        ascending=[False, False, True]
    )
    return sub.iloc[0]

def metric_or_dash(row, metric):
    if row is None:
        return "---"
    return f"{row[metric]:.3f}"

def latex_method_label(row, fallback):
    if row is None:
        return fallback
    name = row["ansatz"]
    name = name.replace("W1+W2", "W$_1$W$_2$")
    name = name.replace("_", r"\_")
    return name

def collect_dataset_block(df: pd.DataFrame, dataset_name: str):
    rows = []
    sub = df[df["dataset_name"] == dataset_name].copy()
    for q in sorted(sub["input_dim"].unique()):
        g = sub[sub["input_dim"] == q].copy()
        rows.append({
            "q": int(q),
            "hea": best_row(g, is_hea),
            "wi": best_row(g, is_wi),
            "mps": best_row(g, is_mps),
            "ttn": best_row(g, is_ttn),
            "random": best_row(g, is_random),
        })
    return rows

def build_dataset_table(dataset_name: str, label: str, caption: str) -> str:
    blocks = collect_dataset_block(summary5, dataset_name)

    lines = []
    lines.append(r"\begin{table}[t]")
    lines.append(r"\centering")
    lines.append(rf"\caption{{{caption}}}")
    lines.append(rf"\label{{{label}}}")
    lines.append(r"\renewcommand{\arraystretch}{1.15}")
    lines.append(r"\setlength{\tabcolsep}{5pt}")
    lines.append(r"\footnotesize")
    lines.append(r"\begin{tabular}{lcccc}")
    lines.append(r"\toprule")
    lines.append(r"\textbf{Method} & \textbf{Acc.} & \textbf{Prec.} & \textbf{Rec.} & \textbf{F1} \\")
    lines.append(r"\midrule")

    first_block = True
    for block in blocks:
        if not first_block:
            lines.append(r"\midrule")
        first_block = False

        q = block["q"]
        lines.append(rf"\multicolumn{{5}}{{l}}{{\textbf{{{q} qubits}}}} \\")
        lines.append(r"\midrule")

        ordered = [
            ("HEA (y, z)", block["hea"]),
            (latex_method_label(block["wi"], "WI-PQC"), block["wi"]),
            (latex_method_label(block["mps"], "MPS"), block["mps"]),
            (latex_method_label(block["ttn"], "TTN"), block["ttn"]),
            ("Random PQC", block["random"]),
        ]

        for method_name, row in ordered:
            lines.append(
                rf"{method_name} & "
                rf"{metric_or_dash(row, 'acc_mean')} & "
                rf"{metric_or_dash(row, 'prec_mean')} & "
                rf"{metric_or_dash(row, 'rec_mean')} & "
                rf"{metric_or_dash(row, 'f1_mean')} \\"
            )

    lines.append(r"\bottomrule")
    lines.append(r"\end{tabular}")
    lines.append(r"\end{table}")

    return "\n".join(lines)

# ============================================================
# GENERATE TABLES
# ============================================================

iris_table = build_dataset_table(
    dataset_name="iris",
    label="tab:iris-summary",
    caption="Iris benchmark results. Entries are mean test accuracy, precision, recall, and F1 over 5 seeds."
)

wine_table = build_dataset_table(
    dataset_name="wine",
    label="tab:wine-summary",
    caption="Wine benchmark results. Entries are mean test accuracy, precision, recall, and F1 over 5 seeds."
)

diabetes_table = build_dataset_table(
    dataset_name="diabetes",
    label="tab:diabetes-summary",
    caption="Diabetes benchmark results. Entries are mean test accuracy, precision, recall, and F1 over 5 seeds."
)

print("===== IRIS TABLE =====")
print(iris_table)
print("\n")

print("===== WINE TABLE =====")
print(wine_table)
print("\n")

print("===== DIABETES TABLE =====")
print(diabetes_table)
print("\n")

# Save
out_dir = summary5_fp.parent
(out_dir / "latex_iris_summary_explicit.tex").write_text(iris_table)
(out_dir / "latex_wine_summary_explicit.tex").write_text(wine_table)
(out_dir / "latex_diabetes_summary_explicit.tex").write_text(diabetes_table)

print("Saved:")
print(out_dir / "latex_iris_summary_explicit.tex")
print(out_dir / "latex_wine_summary_explicit.tex")
print(out_dir / "latex_diabetes_summary_explicit.tex")

===== IRIS TABLE =====
\begin{table}[t]
\centering
\caption{Iris benchmark results. Entries are mean test accuracy, precision, recall, and F1 over 5 seeds.}
\label{tab:iris-summary}
\renewcommand{\arraystretch}{1.15}
\setlength{\tabcolsep}{5pt}
\footnotesize
\begin{tabular}{lcccc}
\toprule
\textbf{Method} & \textbf{Acc.} & \textbf{Prec.} & \textbf{Rec.} & \textbf{F1} \\
\midrule
\multicolumn{5}{l}{\textbf{4 qubits}} \\
\midrule
HEA & 0.607 & 0.644 & 0.607 & 0.599 \\
Best WI-PQC & 0.633 & 0.641 & 0.633 & 0.629 \\
Best TN baseline & 0.553 & 0.561 & 0.553 & 0.550 \\
Random PQC & 0.340 & 0.334 & 0.340 & 0.323 \\
\bottomrule
\end{tabular}
\end{table}


===== WINE TABLE =====
\begin{table}[t]
\centering
\caption{Wine benchmark results. Entries are mean test accuracy, precision, recall, and F1 over 5 seeds.}
\label{tab:wine-summary}
\renewcommand{\arraystretch}{1.15}
\setlength{\tabcolsep}{5pt}
\footnotesize
\begin{tabular}{lcccc}
\toprule
\textbf{Method} & \textbf{Acc.} & \textbf{Prec.} & \t

In [22]:
import pandas as pd
from pathlib import Path


merged_fp = Path("/Users/larryhh/Documents/PhD/Projects/weight_matrix_informed_circuit_design/results/merged_analysis/wi_experiment_merged.csv")
df = pd.read_csv(merged_fp)

cdf = df[df["model_type"] == "classical"].copy()

classical_summary = (
    cdf.groupby(["dataset_name", "input_dim"])
       .agg(
           n_runs=("test_f1", "count"),
           acc_mean=("test_acc", "mean"),
           acc_std=("test_acc", "std"),
           prec_mean=("test_prec", "mean"),
           prec_std=("test_prec", "std"),
           rec_mean=("test_rec", "mean"),
           rec_std=("test_rec", "std"),
           f1_mean=("test_f1", "mean"),
           f1_std=("test_f1", "std"),
       )
       .reset_index()
       .sort_values(["dataset_name", "input_dim"])
)

print(classical_summary.to_string(index=False))

for _, r in classical_summary.iterrows():
    print(
        f'{r["dataset_name"].capitalize()} & {int(r["input_dim"])} & '
        f'{r["acc_mean"]:.3f} & {r["prec_mean"]:.3f} & {r["rec_mean"]:.3f} & {r["f1_mean"]:.3f} \\\\'
    )


dataset_name  input_dim  n_runs  acc_mean  acc_std  prec_mean  prec_std  rec_mean  rec_std  f1_mean   f1_std
    diabetes          4       5   0.68700 0.036597    0.44770  0.255556   0.48086 0.271018 0.461980 0.259492
    diabetes          6       5   0.73896 0.044608    0.63664  0.095277   0.62004 0.063203 0.622500 0.040202
    diabetes          8       5   0.71560 0.014092    0.59098  0.022703   0.58308 0.087989 0.583700 0.042232
        iris          4       4   0.94165 0.041948    0.94780  0.040038   0.94165 0.041948 0.941275 0.042127
        wine          4       5   0.92224 0.049691    0.93160  0.043134   0.92578 0.045130 0.924920 0.046843
        wine          6       5   0.95556 0.037245    0.95836  0.037216   0.95864 0.032429 0.957600 0.034812
        wine          8       5   0.96664 0.023259    0.96860  0.023988   0.96896 0.020484 0.967700 0.022234
Diabetes & 4 & 0.687 & 0.448 & 0.481 & 0.462 \\
Diabetes & 6 & 0.739 & 0.637 & 0.620 & 0.622 \\
Diabetes & 8 & 0.716 & 0.591 & 0

In [26]:
# summary5 view for all anstaz with wi-pqc in it

summary5[summary5["ansatz"].str.contains("WI-PQC", na=False)].to_csv(Path("/Users/larryhh/Documents/PhD/Projects/weight_matrix_informed_circuit_design/results/merged_analysis/wi_pqc_only_summary5.csv"), index=False)

In [45]:
# summary5[summary5["ansatz"].str.contains("WI-PQC", na=False)]
_df = summary5[summary5["ansatz"].str.strip() == "WI-PQC (W1+W2+E, svd)"]
_df

,dataset_name,input_dim,ansatz,n_runs,acc_mean,acc_std,prec_mean,prec_std,rec_mean,rec_std,f1_mean,f1_std,loss_mean,loss_std,time_mean,time_std
9,diabetes,4,"WI-PQC (W1+W2+E, svd)",5,0.62600,0.066948,0.45642,0.125483,0.34586,0.064619,0.39218,0.085952,0.61400,0.042474,50.65168,0.659674
23,diabetes,6,"WI-PQC (W1+W2+E, svd)",5,0.65454,0.011577,0.49628,0.035066,0.25968,0.086581,0.33554,0.078929,0.64604,0.007813,221.59500,168.215397
43,diabetes,8,"WI-PQC (W1+W2+E, svd)",5,0.60782,0.041731,0.39714,0.074643,0.23300,0.065679,0.28798,0.058875,0.66812,0.009441,430.84154,6.912288
45,iris,4,"WI-PQC (W1+W2+E, svd)",5,0.63334,0.040825,0.64134,0.036594,0.63334,0.040825,0.62900,0.037750,0.93464,0.080559,9.75578,0.179841
62,wine,4,"WI-PQC (W1+W2+E, svd)",5,0.65554,0.050449,0.66226,0.062763,0.64858,0.059020,0.64108,0.057920,0.86808,0.030387,11.73034,0.158901
79,wine,6,"WI-PQC (W1+W2+E, svd)",5,0.46112,0.057609,0.46360,0.064718,0.48672,0.067311,0.45704,0.049714,1.05320,0.019946,51.56586,39.984464
95,wine,8,"WI-PQC (W1+W2+E, svd)",5,0.40556,0.063963,0.39700,0.070518,0.40112,0.052494,0.39228,0.064169,1.08620,0.007285,101.14654,1.637293


In [51]:
_df[_df["dataset_name"] == "diabetes"][['input_dim','acc_mean', 'prec_mean', 'rec_mean', 'f1_mean']]

,input_dim,acc_mean,prec_mean,rec_mean,f1_mean
9,4,0.62600,0.45642,0.34586,0.39218
23,6,0.65454,0.49628,0.25968,0.33554
43,8,0.60782,0.39714,0.23300,0.28798


In [44]:
# target_ansatz = "WI-PQC (W1+W2+E, svd)"
target_ansatz = "WI-PQC (W+E, svd)"

wipqc = summary5[summary5["ansatz"].str.contains("WI-PQC", na=False)].copy()
wipqc["ansatz"] = wipqc["ansatz"].str.strip()

group_cols = ["dataset_name", "input_dim"]

# ranks within each (dataset_name, input_dim) group
wipqc["acc_rank"] = (
    wipqc.groupby(group_cols)["acc_mean"]
    .rank(method="min", ascending=False)
)

wipqc["f1_rank"] = (
    wipqc.groupby(group_cols)["f1_mean"]
    .rank(method="min", ascending=False)
)

def summarize_group(g):
    target = g[g["ansatz"] == target_ansatz]
    best_acc = g.loc[[g["acc_mean"].idxmax()]].assign(selection="best_acc")
    worst_acc = g.loc[[g["acc_mean"].idxmin()]].assign(selection="worst_acc")
    best_f1 = g.loc[[g["f1_mean"].idxmax()]].assign(selection="best_f1")
    worst_f1 = g.loc[[g["f1_mean"].idxmin()]].assign(selection="worst_f1")

    parts = [
        target.assign(selection="target"),
        best_acc,
        worst_acc,
        best_f1,
        worst_f1,
    ]

    return pd.concat(parts)

result = (
    wipqc.groupby(group_cols, group_keys=False)
    .apply(summarize_group)
    .reset_index(drop=True)
)

result = result[
    [
        "selection",
        "dataset_name",
        "input_dim",
        "ansatz",
        "acc_mean",
        "acc_rank",
        "f1_mean",
        "f1_rank",
    ]
]

result

,selection,dataset_name,input_dim,ansatz,acc_mean,acc_rank,f1_mean,f1_rank
0,target,diabetes,4,"WI-PQC (W+E, svd)",0.65196,7.0,0.42858,1.0
1,best_acc,diabetes,4,"WI-PQC (E) (mwst, cosine_norm)",0.67404,1.0,0.42378,2.0
2,worst_acc,diabetes,4,"WI-PQC (W1+W2+E, svd)",0.62600,8.0,0.39218,8.0
3,best_f1,diabetes,4,"WI-PQC (W+E, svd)",0.65196,7.0,0.42858,1.0
4,worst_f1,diabetes,4,"WI-PQC (W1+W2+E, svd)",0.62600,8.0,0.39218,8.0
5,target,diabetes,6,"WI-PQC (W+E, svd)",0.64156,3.0,0.32074,5.0
6,best_acc,diabetes,6,"WI-PQC (E) (greedy, cosine_norm)",0.65842,1.0,0.40830,1.0
7,worst_acc,diabetes,6,"WI-PQC (E) (mwst, cosine_norm)",0.59610,8.0,0.32544,4.0
8,best_f1,diabetes,6,"WI-PQC (E) (greedy, cosine_norm)",0.65842,1.0,0.40830,1.0
9,worst_f1,diabetes,6,"WI-PQC (W1+W2+E, chunking)",0.60518,7.0,0.30344,8.0


In [40]:
wipqc.groupby("ansatz")[["acc_rank", "f1_rank"]].mean().sort_values("acc_rank")

,acc_rank,f1_rank
ansatz,,
"WI-PQC (W1+W2+E, svd)",3.714286,4.000000
"WI-PQC (W+E, svd)",3.857143,4.000000
WI-PQC (E) (greedy),4.142857,4.714286
"WI-PQC (E) (greedy, cosine_norm)",4.285714,4.142857
"WI-PQC (W1+W2+E, chunking)",4.285714,4.571429
"WI-PQC (E) (mwst, cosine_norm)",4.428571,4.857143
"WI-PQC (W+E, chunking)",4.714286,4.571429
"WI-PQC (E) (greedy, top_half)",6.142857,5.142857


In [42]:
result.to_csv(Path("/Users/larryhh/Documents/PhD/Projects/weight_matrix_informed_circuit_design/results/merged_analysis/wipqc_ranking_analysis.csv"), index=False)

In [43]:
# Count how often each ansatz finishes in the top 3
top3_acc = (
    wipqc[wipqc["acc_rank"] <= 3]
    .groupby("ansatz")
    .size()
    .rename("top3_acc_count")
    .sort_values(ascending=False)
)

top3_f1 = (
    wipqc[wipqc["f1_rank"] <= 3]
    .groupby("ansatz")
    .size()
    .rename("top3_f1_count")
    .sort_values(ascending=False)
)

top3 = pd.concat([top3_acc, top3_f1], axis=1).fillna(0).astype(int)
top3["total_top3"] = top3["top3_acc_count"] + top3["top3_f1_count"]
top3.sort_values("total_top3", ascending=False)

,top3_acc_count,top3_f1_count,total_top3
ansatz,,,
"WI-PQC (W+E, svd)",5,3,8
"WI-PQC (W1+W2+E, svd)",3,4,7
WI-PQC (E) (greedy),4,2,6
"WI-PQC (E) (greedy, cosine_norm)",3,3,6
"WI-PQC (E) (mwst, cosine_norm)",3,2,5
"WI-PQC (W1+W2+E, chunking)",2,2,4
"WI-PQC (W+E, chunking)",1,3,4
"WI-PQC (E) (greedy, top_half)",1,2,3


In [57]:
import pandas as pd
from pathlib import Path

merged_fp = Path("/Users/larryhh/Documents/PhD/Projects/weight_matrix_informed_circuit_design/results/merged_analysis/wi_experiment_merged.csv")
df = pd.read_csv(merged_fp)

keep_dims = [4, 6, 8]

# --- Classical timings ---
cdf = df[(df["model_type"] == "classical") & (df["input_dim"].isin(keep_dims))].copy()
cdf["total_time"] = cdf["average_time_per_epoch"] * cdf["epochs"]

classical_timing = (
    cdf.groupby(["dataset_name", "input_dim"])
       .agg(
           n_runs=("total_time", "count"),
           time_per_epoch_mean=("average_time_per_epoch", "mean"),
           time_per_epoch_std=("average_time_per_epoch", "std"),
           total_time_mean=("total_time", "mean"),
           total_time_std=("total_time", "std"),
           epochs=("epochs", "first"),
       )
       .reset_index()
       .sort_values(["dataset_name", "input_dim"])
)

print("===== CLASSICAL TIMINGS =====")
print(classical_timing.to_string(index=False))
print()

# --- WI-PQC timings (averaged across all WI-PQC variants) ---
wdf = df[
    (df["model_type"] == "quantum") &
    (df["ansatz"].str.contains("WI-PQC", na=False)) &
    (df["input_dim"].isin(keep_dims))
].copy()
wdf["total_time"] = wdf["average_time_per_epoch"] * wdf["epochs"]

wdf["z"] = wdf.groupby(["dataset_name", "input_dim"])["total_time"].transform(
    lambda x: (x - x.mean()) / x.std()
)

outliers = wdf[wdf["z"].abs() > 2]
print(f"Removing {len(outliers)} outlier rows:")
print(outliers[["dataset_name", "input_dim", "ansatz", "seed", "total_time", "z"]].to_string(index=False))
print()

wdf_clean = wdf[wdf["z"].abs() <= 2].copy()


wipqc_timing = (
    wdf_clean.groupby(["dataset_name", "input_dim"])
       .agg(
           n_runs=("total_time", "count"),
           time_per_epoch_mean=("average_time_per_epoch", "mean"),
           time_per_epoch_std=("average_time_per_epoch", "std"),
           total_time_mean=("total_time", "mean"),
           total_time_std=("total_time", "std"),
           epochs=("epochs", "first"),
       )
       .reset_index()
       .sort_values(["dataset_name", "input_dim"])
)

print("===== WI-PQC TIMINGS (ALL VARIANTS AVERAGED) =====")
print(wipqc_timing.to_string(index=False))
print()

# --- Combined: total WI-PQC pipeline = classical pretrain + quantum train ---
combined = classical_timing[["dataset_name", "input_dim", "total_time_mean", "epochs"]].rename(
    columns={"total_time_mean": "classical_total", "epochs": "classical_epochs"}
).merge(
    wipqc_timing[["dataset_name", "input_dim", "total_time_mean", "epochs"]].rename(
        columns={"total_time_mean": "wipqc_total", "epochs": "quantum_epochs"}
    ),
    on=["dataset_name", "input_dim"],
    how="inner",
)
combined["pipeline_total"] = combined["classical_total"] + combined["wipqc_total"]

print("===== FULL WI-PQC PIPELINE TIME (classical + quantum) =====")
print(combined.to_string(index=False))

===== CLASSICAL TIMINGS =====
dataset_name  input_dim  n_runs  time_per_epoch_mean  time_per_epoch_std  total_time_mean  total_time_std  epochs
    diabetes          4       5              0.01316            0.001443            1.316        0.144326     100
    diabetes          6       5              0.01348            0.001938            1.348        0.193830     100
    diabetes          8       5              0.03950            0.002591            3.950        0.259133     100
        iris          4       4              0.00505            0.000768            0.505        0.076811     100
        wine          4       5              0.00538            0.000522            0.538        0.052154     100
        wine          6       5              0.00550            0.000866            0.550        0.086603     100
        wine          8       5              0.02036            0.000814            2.036        0.081425     100

Removing 8 outlier rows:
dataset_name  input_dim         

In [58]:
combined

,dataset_name,input_dim,classical_total,classical_epochs,wipqc_total,quantum_epochs,pipeline_total
0,diabetes,4,1.316,100,1349.361243,30,1350.677243
1,diabetes,6,1.348,100,3663.490500,30,3664.838500
2,diabetes,8,3.950,100,10024.610400,30,10028.560400
3,iris,4,0.505,100,225.009375,30,225.514375
4,wine,4,0.538,100,271.520550,30,272.058550
5,wine,6,0.550,100,1428.163500,30,1428.713500
6,wine,8,2.036,100,2340.014692,30,2342.050692


In [61]:
combined["classical_total_min"] = combined["classical_total"] / 60
combined["wipqc_total_min"] = combined["wipqc_total"] / 60
combined["pipeline_total_min"] = combined["pipeline_total"] / 60

combined[["dataset_name", "input_dim", "classical_total_min", "wipqc_total_min", "pipeline_total_min"]]

,dataset_name,input_dim,classical_total_min,wipqc_total_min,pipeline_total_min
0,diabetes,4,0.021933,22.489354,22.511287
1,diabetes,6,0.022467,61.058175,61.080642
2,diabetes,8,0.065833,167.076840,167.142673
3,iris,4,0.008417,3.750156,3.758573
4,wine,4,0.008967,4.525342,4.534309
5,wine,6,0.009167,23.802725,23.811892
6,wine,8,0.033933,39.000245,39.034178
